# Lab 20 — Convolutional Models for Sequence Data with TensorFlow

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-20-convolutional-models-sequences-lab.ipynb)

This lab is designed for independent study. Read the background sections before running the code. The goal is not only to train a neural network, but also to understand why convolutional models are useful for time series and sequence data.

## Learning goals

By the end of this lab, you should be able to:

1. Explain how a one-dimensional convolution scans a sequence.
2. Build supervised learning windows for sequence forecasting.
3. Train a TensorFlow/Keras Conv1D forecasting model.
4. Explain causal convolution and why it matters for forecasting.
5. Explain dilation and receptive field in a temporal convolutional network.
6. Compare a CNN/TCN forecast with simple baselines.
7. Diagnose common leakage and modeling mistakes.

## How this lab fits the course

Classical AR and ARMA models forecast from a finite or infinite linear combination of past values. Convolutional models are a modern machine-learning extension of this idea: they learn filters from data. A convolutional filter can detect local patterns such as spikes, changes in slope, periodic fragments, or short motifs. Dilated causal convolutions allow the model to look farther back without using too many layers.

## 1. Mathematical background

### 1.1 One-dimensional convolution as a learned local filter

For a univariate sequence

$$
(x_1,x_2,\ldots,x_T),
$$

an ordinary length-$K$ finite impulse-response filter forms quantities like

$$
y_t = \sum_{j=0}^{K-1} w_j x_{t-j}.
$$

In signal processing this is often called a convolution. In many deep-learning libraries, including TensorFlow/Keras, `Conv1D` computes a closely related operation called cross-correlation. The difference is the order of the filter coefficients. Since the coefficients are learned, the distinction is usually not important for training; it is important only when we manually compare formulas.

### 1.2 Causal convolution

For forecasting, a prediction made at time $t$ should not use values from times after $t$. A causal convolution respects the time direction:

$$
y_t = b + \sum_{j=0}^{K-1} w_j x_{t-j}.
$$

This is similar in spirit to an autoregressive model, but the filter weights are learned jointly with nonlinear activations and multiple layers.

### 1.3 Dilated convolution

A dilated convolution skips inputs at a regular spacing. With dilation $d$,

$$
y_t = b + \sum_{j=0}^{K-1} w_j x_{t-dj}.
$$

For example, a kernel of size $3$ with dilation $4$ uses lags $0,4,8$. Stacking layers with dilation rates $1,2,4,8,\ldots$ gives a rapidly growing receptive field.

### 1.4 Receptive field

The receptive field is the set of input time points that can influence an output. For a stack of causal convolutions with kernel size $K$ and dilation rates $d_1,d_2,\ldots,d_L$, a simple receptive-field formula is

$$
R = 1 + (K-1)\sum_{\ell=1}^L d_\ell.
$$

If each residual block has two convolutional layers with the same dilation, then

$$
R = 1 + 2(K-1)\sum_{\ell=1}^L d_\ell.
$$

A model cannot use information outside its receptive field. This is why receptive field should be compared with the input window length and with the time scale of the patterns in the data.

## 2. Programming setup

This lab uses TensorFlow/Keras. Google Colab normally has TensorFlow installed. If you run this locally and do not have TensorFlow, install it first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(7339)
tf.keras.utils.set_random_seed(7339)

# Keep training reasonably reproducible and Colab-friendly.
try:
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except RuntimeError:
    # TensorFlow threading settings may already be initialized in some environments.
    pass

print("TensorFlow version:", tf.__version__)

## 3. Build a synthetic sequence with local patterns

We will create a univariate sequence with three ingredients:

1. a smooth seasonal component,
2. short local pulses,
3. random noise.

The local pulses are useful because convolutional filters are designed to detect short local patterns.

In [ ]:
rng = np.random.default_rng(7339)

n = 2400
t = np.arange(n)

seasonal = 0.7 * np.sin(2 * np.pi * t / 50) + 0.25 * np.sin(2 * np.pi * t / 13)
slow = 0.002 * t
noise = rng.normal(0, 0.18, size=n)

# Add local pulse motifs at irregular times.
pulse = np.zeros(n)
motif = np.array([0.0, 0.25, 0.70, 1.10, 0.70, 0.25, 0.0])
starts = np.arange(80, n - len(motif), 145)
for s in starts:
    jitter = rng.integers(-12, 13)
    idx = s + jitter
    if 0 <= idx and idx + len(motif) <= n:
        pulse[idx:idx + len(motif)] += motif

values = seasonal + slow + pulse + noise

series = pd.DataFrame({
    "time": t,
    "value": values,
    "seasonal": seasonal,
    "pulse": pulse
})

series.head()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(series["time"], series["value"], label="observed sequence")
plt.plot(series["time"], series["seasonal"] + series["pulse"], label="signal without noise", alpha=0.8)
plt.xlabel("time")
plt.ylabel("value")
plt.title("Synthetic sequence with smooth seasonality and local pulse motifs")
plt.legend()
plt.show()

### Checkpoint 1

Before modeling, answer these questions:

1. What patterns are global or long-range?
2. What patterns are local?
3. Why might a convolutional model be more natural than a fully connected MLP for detecting the pulse motif?

## 4. Chronological train, validation, and test split

For sequence forecasting, random splitting is usually wrong because it mixes future information into the training set. We split the raw sequence chronologically first, and then build windows within each split.

In [ ]:
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train_values = values[:train_end]
val_values = values[train_end:val_end]
test_values = values[val_end:]

print("Train length:", len(train_values))
print("Validation length:", len(val_values))
print("Test length:", len(test_values))

plt.figure(figsize=(11, 4))
plt.plot(np.arange(n), values, label="full series")
plt.axvline(train_end, linestyle="--", label="train/validation boundary")
plt.axvline(val_end, linestyle="--", label="validation/test boundary")
plt.xlabel("time")
plt.ylabel("value")
plt.title("Chronological split")
plt.legend()
plt.show()

## 5. Scale using training data only

Scaling should be learned from the training set only. Using the whole sequence to compute the mean and standard deviation would leak information from the validation and test periods.

In [ ]:
train_mean = train_values.mean()
train_std = train_values.std()

train_scaled = (train_values - train_mean) / train_std
val_scaled = (val_values - train_mean) / train_std
test_scaled = (test_values - train_mean) / train_std

print("Training mean used for scaling:", round(train_mean, 4))
print("Training std used for scaling:", round(train_std, 4))

## 6. Build supervised learning windows

A forecasting neural network does not automatically know what the input and label are. We must convert the sequence into pairs:

$$
\text{input window } (x_t,\ldots,x_{t+L-1})
\quad \longrightarrow \quad
\text{label window } (x_{t+L},\ldots,x_{t+L+H-1}).
$$

Here $L$ is the input width and $H$ is the forecast horizon length.

In [ ]:
def make_windows(sequence, input_width, label_width):
    """Convert a 1D sequence into input and label windows.

    Returns
    -------
    X : array with shape (number_of_windows, input_width, 1)
    y : array with shape (number_of_windows, label_width)
    """
    sequence = np.asarray(sequence, dtype=np.float32)
    total_width = input_width + label_width
    X_list = []
    y_list = []

    for start in range(len(sequence) - total_width + 1):
        end_input = start + input_width
        end_label = end_input + label_width
        X_list.append(sequence[start:end_input])
        y_list.append(sequence[end_input:end_label])

    X = np.array(X_list, dtype=np.float32)[..., None]
    y = np.array(y_list, dtype=np.float32)
    return X, y

input_width = 72
label_width = 12

X_train, y_train = make_windows(train_scaled, input_width, label_width)
X_val, y_val = make_windows(val_scaled, input_width, label_width)
X_test, y_test = make_windows(test_scaled, input_width, label_width)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
def plot_window(X, y, index=0):
    x_window = X[index, :, 0]
    y_window = y[index]

    plt.figure(figsize=(9, 4))
    plt.plot(np.arange(input_width), x_window, marker="o", label="input window")
    plt.plot(np.arange(input_width, input_width + label_width), y_window, marker="x", label="label window")
    plt.axvline(input_width - 0.5, linestyle="--")
    plt.xlabel("relative time")
    plt.ylabel("scaled value")
    plt.title("One supervised learning example")
    plt.legend()
    plt.show()

plot_window(X_train, y_train, index=30)

### Checkpoint 2

Explain why the input array has shape `(number_of_windows, input_width, 1)`. What does the final dimension represent?

## 7. TensorFlow datasets

TensorFlow datasets handle batching, shuffling, and prefetching efficiently. We shuffle only the training windows. Validation and test windows are left in chronological order for easier interpretation.

In [ ]:
batch_size = 32

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=len(X_train), seed=7339)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

for xb, yb in train_ds.take(1):
    print("Batch X shape:", xb.shape)
    print("Batch y shape:", yb.shape)

## 8. Baseline forecast

A neural network should beat simple baselines. We will use a last-value baseline:

$$
\hat{x}_{t+h} = x_t, \qquad h=1,\ldots,H.
$$

This is simple but often surprisingly strong.

In [ ]:
def last_value_baseline(X, label_width):
    last = X[:, -1, 0]
    return np.repeat(last[:, None], label_width, axis=1)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

baseline_val = last_value_baseline(X_val, label_width)
baseline_test = last_value_baseline(X_test, label_width)

print("Validation baseline MAE:", round(mae(y_val, baseline_val), 4))
print("Validation baseline RMSE:", round(rmse(y_val, baseline_val), 4))
print("Test baseline MAE:", round(mae(y_test, baseline_test), 4))
print("Test baseline RMSE:", round(rmse(y_test, baseline_test), 4))

## 9. A first Conv1D forecasting model

This model uses causal convolutional layers. Causal padding prevents the convolution from using future values inside the input window. The final dense layer outputs the next `label_width` values.

In [ ]:
def build_cnn_model(input_width, label_width):
    model = keras.Sequential([
        keras.Input(shape=(input_width, 1)),
        layers.Conv1D(filters=32, kernel_size=5, padding="causal", activation="relu"),
        layers.Conv1D(filters=32, kernel_size=5, padding="causal", activation="relu"),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation="relu"),
        layers.Dense(label_width)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

cnn_model = build_cnn_model(input_width, label_width)
cnn_model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_cnn = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
def plot_history(history, title):
    hist = pd.DataFrame(history.history)
    plt.figure(figsize=(8, 4))
    plt.plot(hist["loss"], label="training loss")
    plt.plot(hist["val_loss"], label="validation loss")
    plt.xlabel("epoch")
    plt.ylabel("MSE loss")
    plt.title(title)
    plt.legend()
    plt.show()

plot_history(history_cnn, "Conv1D training history")

In [ ]:
cnn_val_pred = cnn_model.predict(X_val, verbose=0)
cnn_test_pred = cnn_model.predict(X_test, verbose=0)

print("CNN validation MAE:", round(mae(y_val, cnn_val_pred), 4))
print("CNN validation RMSE:", round(rmse(y_val, cnn_val_pred), 4))
print("CNN test MAE:", round(mae(y_test, cnn_test_pred), 4))
print("CNN test RMSE:", round(rmse(y_test, cnn_test_pred), 4))

print("Baseline test MAE:", round(mae(y_test, baseline_test), 4))
print("Baseline test RMSE:", round(rmse(y_test, baseline_test), 4))

In [ ]:
def plot_forecast_example(X, y, pred, index=0, title="Forecast example"):
    x_window = X[index, :, 0]
    y_window = y[index]
    pred_window = pred[index]

    plt.figure(figsize=(9, 4))
    plt.plot(np.arange(input_width), x_window, label="input")
    plt.plot(np.arange(input_width, input_width + label_width), y_window, marker="o", label="true future")
    plt.plot(np.arange(input_width, input_width + label_width), pred_window, marker="x", label="forecast")
    plt.axvline(input_width - 0.5, linestyle="--")
    plt.xlabel("relative time")
    plt.ylabel("scaled value")
    plt.title(title)
    plt.legend()
    plt.show()

plot_forecast_example(X_test, y_test, cnn_test_pred, index=40, title="Conv1D multi-step forecast")

### Checkpoint 3

Compare the CNN forecast with the last-value baseline. Does the CNN improve both MAE and RMSE? If it improves one metric but not the other, what might that mean?

## 10. What does TensorFlow Conv1D compute?

TensorFlow/Keras `Conv1D` computes cross-correlation rather than a flipped mathematical convolution. The following small example verifies the operation directly.

In [ ]:
sample = np.array([1, 2, 3, 4, 5], dtype=np.float32).reshape(1, 5, 1)
kernel = np.array([0.2, 0.5, 0.3], dtype=np.float32).reshape(3, 1, 1)

conv = layers.Conv1D(filters=1, kernel_size=3, padding="valid", use_bias=False)
_ = conv(sample)
conv.set_weights([kernel])

tf_result = conv(sample).numpy().ravel()
manual_result = np.array([
    np.dot(sample.ravel()[i:i+3], kernel.ravel())
    for i in range(3)
])

print("TensorFlow Conv1D result:", tf_result)
print("Manual cross-correlation result:", manual_result)
print("They match:", np.allclose(tf_result, manual_result))

## 11. Receptive field calculation

The receptive field tells us how many past time points can influence the model output. For two causal convolution layers with kernel size $5$ and dilation $1$, the receptive field is

$$
R = 1 + 2(5-1) = 9.
$$

That is much smaller than the input window length $72$. A temporal convolutional network increases the receptive field using dilation.

In [ ]:
def receptive_field(kernel_size, dilations, convs_per_block=1):
    return 1 + convs_per_block * (kernel_size - 1) * sum(dilations)

print("CNN receptive field:", receptive_field(kernel_size=5, dilations=[1, 1], convs_per_block=1))
print("TCN receptive field with dilations 1,2,4,8:", receptive_field(kernel_size=3, dilations=[1, 2, 4, 8], convs_per_block=2))
print("Input window length:", input_width)

## 12. Temporal convolutional network with dilated causal convolutions

A temporal convolutional network, or TCN, usually uses:

1. causal convolutions,
2. dilation rates such as $1,2,4,8$,
3. residual connections,
4. nonlinear activations.

The residual connection makes the model easier to train because information and gradients can flow around the convolutional block.

In [ ]:
def residual_tcn_block(x, filters, kernel_size, dilation_rate, dropout_rate=0.0):
    y = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(x)
    y = layers.Dropout(dropout_rate)(y)
    y = layers.Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        padding="causal",
        dilation_rate=dilation_rate,
        activation="relu"
    )(y)

    if x.shape[-1] != filters:
        x = layers.Conv1D(filters=filters, kernel_size=1, padding="same")(x)

    return layers.Add()([x, y])


def build_tcn_model(input_width, label_width):
    inputs = keras.Input(shape=(input_width, 1))
    x = inputs

    for dilation in [1, 2, 4, 8]:
        x = residual_tcn_block(
            x,
            filters=32,
            kernel_size=3,
            dilation_rate=dilation,
            dropout_rate=0.05
        )

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(label_width)(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

tcn_model = build_tcn_model(input_width, label_width)
tcn_model.summary()

In [ ]:
early_stop_tcn = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history_tcn = tcn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[early_stop_tcn],
    verbose=1
)

In [ ]:
plot_history(history_tcn, "TCN training history")

tcn_val_pred = tcn_model.predict(X_val, verbose=0)
tcn_test_pred = tcn_model.predict(X_test, verbose=0)

results = pd.DataFrame({
    "model": ["last-value baseline", "Conv1D", "TCN"],
    "validation_MAE": [mae(y_val, baseline_val), mae(y_val, cnn_val_pred), mae(y_val, tcn_val_pred)],
    "validation_RMSE": [rmse(y_val, baseline_val), rmse(y_val, cnn_val_pred), rmse(y_val, tcn_val_pred)],
    "test_MAE": [mae(y_test, baseline_test), mae(y_test, cnn_test_pred), mae(y_test, tcn_test_pred)],
    "test_RMSE": [rmse(y_test, baseline_test), rmse(y_test, cnn_test_pred), rmse(y_test, tcn_test_pred)]
})

results

In [ ]:
plot_forecast_example(X_test, y_test, tcn_test_pred, index=40, title="TCN multi-step forecast")

### Checkpoint 4

The TCN has a larger receptive field than the first CNN model. Does that lead to better validation or test error in this synthetic example? Why might a larger receptive field help? Why might it sometimes not help?

## 13. Horizon-wise error analysis

Multi-step forecasts often become less accurate as the horizon increases. We will compute the MAE separately for each forecast step.

In [ ]:
def horizon_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred), axis=0)

h = np.arange(1, label_width + 1)

plt.figure(figsize=(8, 4))
plt.plot(h, horizon_mae(y_test, baseline_test), marker="o", label="baseline")
plt.plot(h, horizon_mae(y_test, cnn_test_pred), marker="o", label="Conv1D")
plt.plot(h, horizon_mae(y_test, tcn_test_pred), marker="o", label="TCN")
plt.xlabel("forecast horizon")
plt.ylabel("test MAE")
plt.title("Horizon-wise forecast error")
plt.legend()
plt.show()

## 14. Inspect learned feature maps

A convolutional layer transforms one input sequence into several feature sequences. Each channel can be interpreted as a learned detector of some local pattern.

The following code extracts activations from the first convolutional layer of the CNN model.

In [ ]:
first_conv_layer = None
for layer in cnn_model.layers:
    if isinstance(layer, layers.Conv1D):
        first_conv_layer = layer
        break

feature_model = keras.Model(inputs=cnn_model.input, outputs=first_conv_layer.output)
activations = feature_model.predict(X_test[:1], verbose=0)[0]

print("Activation array shape:", activations.shape)

plt.figure(figsize=(10, 5))
for channel in range(4):
    plt.plot(activations[:, channel], label=f"channel {channel}")
plt.xlabel("time index inside feature map")
plt.ylabel("activation")
plt.title("First Conv1D layer feature maps for one test window")
plt.legend()
plt.show()

### Checkpoint 5

Look at the feature-map plot. Do some channels activate only during certain local shapes? What would you do to investigate this more carefully?

## 15. Leakage checklist for convolutional sequence models

Before trusting a CNN or TCN forecast, check the following:

1. Was the train/validation/test split chronological?
2. Were scaling parameters learned only from the training set?
3. Were labels constructed from future values relative to the input window?
4. Did any rolling feature use future observations?
5. Does the model use causal padding for forecasting tasks?
6. Was model selection done using validation data rather than test data?
7. Was the final test set evaluated only once after model choices were made?

These checks matter more than model complexity. A leaky simple model can look better than a correct deep model.

## 16. Student exercises

### Exercise 1 — Change the kernel size

Train the Conv1D model with kernel sizes $3$, $5$, and $9$. Compare validation MAE and test MAE. Which kernel size works best on this data?

### Exercise 2 — Change the forecast horizon

Repeat the experiment with `label_width = 1`, `label_width = 6`, and `label_width = 24`. How does the difficulty change?

### Exercise 3 — Change the receptive field

Modify the TCN dilation list from `[1, 2, 4, 8]` to `[1, 2]` and `[1, 2, 4, 8, 16]`. Compare the receptive field and the model performance.

### Exercise 4 — Add multivariate input

Create a second input feature such as a sine/cosine calendar feature. Modify the window function so that `X` has shape `(number_of_windows, input_width, number_of_features)`.

### Exercise 5 — Compare with an MLP

Flatten each input window and train a dense neural network. Does the CNN outperform the MLP? Explain why or why not.

## 17. Mini-project

Choose one of the following.

### Option A — Local pattern forecasting

Modify the synthetic data generator so that local motifs become more frequent or less frequent over time. Compare a last-value baseline, a Conv1D model, and a TCN model.

### Option B — Noisy seasonal forecasting

Remove the pulse motif and keep only seasonality plus noise. Does the CNN still help? Would a classical seasonal model be more appropriate?

### Option C — Anomaly detector from prediction errors

Use the trained CNN or TCN to forecast the next few values. Mark a time point as suspicious if the prediction error is unusually large. Test this idea by injecting artificial spikes into the test set.

Your mini-project report should include:

1. a short description of the data-generating mechanism,
2. plots of the time series and forecast examples,
3. baseline and neural-network metrics,
4. one paragraph explaining what the convolutional model learned,
5. one leakage check.

## 18. AI-assisted study prompts

Use an AI assistant as a critic, not as an authority. Try prompts like these:

1. "Here is my time-series windowing code. Check whether any future information leaks into the training input."
2. "Explain the difference between causal convolution and ordinary same-padding convolution for forecasting."
3. "Given kernel size 3 and dilation rates 1, 2, 4, 8 with two convolutions per block, compute the receptive field and explain each term."
4. "My TCN beats the baseline on validation data but not on test data. List possible reasons and how to check them."
5. "Help me write a short interpretation of a horizon-wise MAE plot for a multi-step forecast."

Always verify the answer by checking the code, the time indices, and the plots yourself.

## 19. Lab summary

In this lab you learned that:

- `Conv1D` learns local filters for sequence data.
- Causal convolution respects the forecasting time direction.
- Dilation increases the receptive field efficiently.
- TCNs combine causal convolutions, dilation, and residual connections.
- Strong baselines and leakage checks are essential.
- Horizon-wise errors often reveal behavior that a single overall metric hides.

Convolutional sequence models are not replacements for statistical thinking. They are powerful tools when combined with careful windowing, validation, diagnostics, and interpretation.